# Verify manifests

Run this interactively on Roihu once `feature/data-preprocessing` has
landed, against the REAL datasets (not synthetic test fixtures). Purpose:
catch anything the unit tests can't -- unit tests use small synthetic
directory trees; this notebook is the first time the manifest builders
run against the actual GRID/LRS3 data.

**What "looks right" means, concretely (come back to this checklist at the
end):**
- Every manifest has the columns in `manifest_builder.MANIFEST_COLUMNS`,
  in that order.
- `sample_id` is unique within each manifest.
- `video_path`, `audio_path`, `landmark_path` all point at files that
  exist (checked below via `check_file_existence`) -- except
  `video_path`/`landmark_path` on `lrs3_test_mattymchen` rows, which are
  expected to be empty strings, not missing files.
- `duration_sec` is a small positive float (a few seconds), not 0 or NaN.
- `transcript` is non-empty, lowercase, and free of the LRS3 `Text:`
  prefix.
- `check_frame_count_vs_duration` and `check_video_fps` both return empty
  DataFrames (zero problem rows).
- `grid_word_segments.csv`'s frame-boundary conversion is directionally
  correct (checked explicitly near the end of this notebook).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # adjust if running from somewhere else
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from fusion_avsr.data.manifest_builder import (
    build_grid_manifest,
    build_grid_word_segments,
    build_lrs3_test_mattymchen_manifest,
    build_lrs3_trainval_manifest,
    check_file_existence,
    check_frame_count_vs_duration,
    check_video_fps,
    _align_units_to_frame,
    _parse_grid_align,
)

## Config -- set these paths for Roihu

Fill in the real dataset roots below (see CLAUDE.md's "Data: verified
structure" section for the exact confirmed paths). `LIMIT` caps how many
clips each manifest builder processes -- keep it small (a handful) for
this first interactive pass; only remove it (`LIMIT = None`) once
everything below looks right and you're ready to build the full
manifests.

In [ ]:
# TODO: fill in for Roihu
LRS3_ROOT = Path("/scratch/project_2020712/datasets/lrs3")
GRID_ROOT = Path("/scratch/project_2020712/datasets/kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data")
GRID_LANDMARKS_ROOT = Path("/scratch/project_2020712/datasets/grid_landmarks")  # from scripts/extract_landmarks_grid.py
MATTYMCHEN_PARQUET_DIR = LRS3_ROOT / "test-mattymchen" / "data"

AUDIO_OUTPUT_DIR = Path("/scratch/project_2020712/datasets/extracted_audio")  # from scripts/extract_audio.sh
MANIFEST_DIR = REPO_ROOT / "manifests"

LIMIT = 5  # a couple of examples from each dataset, not the whole thing

## Build manifests on a small sample first

Using `limit=LIMIT` here samples just a couple of clips from each
dataset, so this cell runs in seconds instead of the tens of minutes a
full build (~65,000 clips total) would take. This is the fast
iteration loop for checking the manifest builders actually work against
real data before committing to a full run.

In [ ]:
lrs3_trainval_manifest = build_lrs3_trainval_manifest(
    LRS3_ROOT, AUDIO_OUTPUT_DIR, limit=LIMIT,
)
lrs3_trainval_manifest.head()

In [ ]:
grid_manifest = build_grid_manifest(
    GRID_ROOT, GRID_LANDMARKS_ROOT, AUDIO_OUTPUT_DIR, limit=LIMIT,
)
grid_manifest.head()

In [ ]:
mattymchen_manifest = build_lrs3_test_mattymchen_manifest(
    MATTYMCHEN_PARQUET_DIR, AUDIO_OUTPUT_DIR, limit=LIMIT,
)
mattymchen_manifest.head()

## Spot-check a few rows by hand

TODO: eyeball a couple of rows from each manifest above -- open a
`video_path` or listen to an `audio_path` directly, and confirm the
`transcript` field actually matches what's said in the clip.

## Re-run the consistency checks

These should all come back empty (zero rows). If they don't, that's a
real problem to chase down before trusting the manifest for anything
downstream.

In [ ]:
print("check_file_existence (lrs3_trainval):")
display(check_file_existence(lrs3_trainval_manifest))

print("check_file_existence (grid):")
display(check_file_existence(grid_manifest))

print("check_file_existence (mattymchen):")
display(check_file_existence(mattymchen_manifest))

In [ ]:
print("check_frame_count_vs_duration (lrs3_trainval):")
display(check_frame_count_vs_duration(lrs3_trainval_manifest))

print("check_frame_count_vs_duration (grid):")
display(check_frame_count_vs_duration(grid_manifest))

In [ ]:
print("check_video_fps (lrs3_trainval):")
display(check_video_fps(lrs3_trainval_manifest))

print("check_video_fps (grid):")
display(check_video_fps(grid_manifest))

## GRID word segments: sanity-check the frame-boundary conversion

`build_grid_word_segments` converts each `.align` timestamp (units of
1/25000 sec) to a frame index via `/25000` then `*25fps`. This is easy to
get subtly wrong (off-by-one, wrong unit, etc.), so before trusting it:
pick one real word segment, and confirm
`(end_frame - start_frame) / 25fps` roughly matches the RAW `.align`
timestamp difference in seconds -- computed independently, without going
through `build_grid_word_segments` itself, as a cross-check.

In [ ]:
grid_word_segments = build_grid_word_segments(GRID_ROOT, limit=LIMIT)
grid_word_segments.head()

In [ ]:
# Pick one word segment and cross-check its frame conversion directly
# against the raw .align file, independently of _align_units_to_frame.
sample_row = grid_word_segments.iloc[0]
sample_id = sample_row["sample_id"]

speaker, clip_id = sample_id.split("_", maxsplit=1)
align_path = GRID_ROOT / f"{speaker}_processed" / "align" / f"{clip_id}.align"

align_rows = _parse_grid_align(align_path)
matching = [r for r in align_rows if r[2] == sample_row["word"]][0]
raw_start_units, raw_end_units, word = matching

raw_duration_sec = (raw_end_units - raw_start_units) / 25000
frame_duration_sec = (sample_row["end_frame"] - sample_row["start_frame"]) / 25

print(f"word={word!r}")
print(f"raw .align duration:  {raw_duration_sec:.3f}s")
print(f"frame-derived duration: {frame_duration_sec:.3f}s")
print(f"difference: {abs(raw_duration_sec - frame_duration_sec):.3f}s (should be well under one frame, ~0.04s)")

## Final checklist

- [ ] All three manifests' columns match `MANIFEST_COLUMNS` exactly.
- [ ] `sample_id` unique within each manifest.
- [ ] `check_file_existence` empty for all three.
- [ ] `check_frame_count_vs_duration` empty for lrs3_trainval and grid.
- [ ] `check_video_fps` empty for lrs3_trainval and grid.
- [ ] GRID word-segment frame conversion cross-check above is within
      ~1 frame (~0.04s) of the raw `.align` duration.
- [ ] Spot-checked transcripts by eye/ear for a couple of rows.

Once every box is checked, remove `limit=LIMIT` (or set `LIMIT = None`)
and re-run to build the full manifests.